# Wave exposure pipeline

Runs the full classification pipeline using the `waves` package.

In [3]:
import geopandas as gpd
import waves
from waves.paths import SRC, CLF, SIE, VRAW, FIN


## 1. Prepare raster data

In [ ]:
waves.prepare.bolge_model_data()


## 2. Inspect NiN 3 classification table

In [5]:
waves.classify.CLASSES


,class_int,trinn,navn_no,navn_en,swm_lower,swm_upper
0,1,0,minimal vannforstyrrelsesintensitet,still water,NaN,1200.0
1,2,a,svært beskyttet,very sheltered,1200.0,4000.0
2,3,b,temmelig beskyttet,moderately sheltered,4000.0,10000.0
3,4,c,litt beskyttet,slightly sheltered,10000.0,50000.0
4,5,d,svakt eksponert,weakly sheltered,50000.0,100000.0
5,6,e,litt eksponert,slightly exposed,100000.0,500000.0
6,7,f,temmelig eksponert,moderately exposed,500000.0,1000000.0
7,8,g,svært eksponert,very exposed,1000000.0,2000000.0
8,9,h,ekstremt eksponert,extremely exposed,2000000.0,4000000.0
9,10,y,disruptivt eksponert,disruptively exposed,4000000.0,NaN


## 3. Classify, sieve, and polygonize

In [ ]:
waves.classify.reclassify_raster(SRC, CLF)
print(f"Classified raster saved: {CLF}")


In [ ]:
waves.classify.sieve_filter(CLF, SIE, threshold=4)
print(f"Sieved raster saved: {SIE}")


In [ ]:
waves.classify.polygonize(SIE, VRAW)
print(f"Raw polygons saved: {VRAW}")


## 4. Subtract land

Load a previously saved checkpoint, or run `subtract_land` to (re)compute it.

In [ ]:
waves.prepare.subtract_land()



## 5. Add NiN class attributes

In [4]:
gdf = gpd.read_file("gdf_grunnlinje.gpkg")
gdf = waves.classify.add_class_attributes(gdf)
gdf.head()


,class_int,geometry,trinn,navn_no,navn_en
0,6,"MULTIPOLYGON (((56516.631 6449106.561, 56516.6...",e,litt eksponert,slightly exposed
1,6,"MULTIPOLYGON (((56541.631 6449131.561, 56541.6...",e,litt eksponert,slightly exposed
2,7,"MULTIPOLYGON (((56541.631 6449156.561, 56541.6...",f,temmelig eksponert,moderately exposed
3,5,"MULTIPOLYGON (((56759 6449181, 56757 6449178, ...",d,svakt eksponert,weakly sheltered
4,5,"MULTIPOLYGON (((56688 6449181.561, 56688 64491...",d,svakt eksponert,weakly sheltered


## 6. Save final output

In [ ]:
gdf.to_file("test.gpkg", driver="GPKG")
